In [5]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import sys
sys.path.insert(0, '/home/kat/Repos/SALSA/')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
seedy = 666
df = pd.read_csv('data/chembl_valid_10to30atoms.csv')

samp_size = 100
df_samp = df.sample(samp_size, random_state=seedy)
df_samp['len'] = df_samp.smiles.apply(lambda x: len(x))
df_samp = df_samp.sort_values(by='len',ignore_index=True)
# df_samp

In [23]:
import numpy as np
from tqdm.notebook import tqdm

from extended_augs_utils import get_ext_augs

iters = 4
max_augs = 10

for aug_iter in range(iters+1):
    
    curr_augset = []
    
    if aug_iter==0:
        for row in df_samp.itertuples():
            anc_idx = row.Index
            src_idx = anc_idx
            smi = row.smiles
            tup = [anc_idx, src_idx, aug_iter, smi]
            curr_augset.append(tup)
            df_augset = pd.DataFrame(curr_augset, columns=['anc_idx', 'src_idx', 'aug_iter', 'smiles'])
        
    else:
        df_src = df_augset[df_augset.aug_iter==(aug_iter-1)]
        print(len(df_src))
        
        for row in tqdm(df_src.itertuples(),total=len(df_src)):
            anc_idx = row.anc_idx
            src_idx = row.Index
            smi = row.smiles
            
            augs = get_ext_augs(smi)
            augs = list(set(augs))
            
            if augs:
                for aug_smi in augs:
                    tup = [anc_idx, src_idx, aug_iter, aug_smi]
                    if tup not in curr_augset:
                        curr_augset.append(tup)
    
        curr_augset = pd.DataFrame(curr_augset, columns=['anc_idx', 'src_idx', 'aug_iter', 'smiles'])
        trunc_idc = []
        for i in range(len(df_samp)):
            df_anc = curr_augset[curr_augset.anc_idx==i]
            df_anc = df_anc.sample(n=10, replace=True)
            idc = df_anc.index.values
            trunc_idc.extend(idc)
        trunc_augset = curr_augset.iloc[trunc_idc]
        
        df_augset = pd.concat([df_augset, trunc_augset], axis=0)

100


  0%|          | 0/100 [00:00<?, ?it/s]

1000


  0%|          | 0/1000 [00:00<?, ?it/s]

1000


  0%|          | 0/1000 [00:00<?, ?it/s]

1000


  0%|          | 0/1000 [00:00<?, ?it/s]

In [15]:
df_augset

,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,C#CCOc1ccc2ccc(=O)oc2c1
1,1,1,0,CCC(N)C(O)c1cc(OC)c(Br)cc1OC
2,2,2,0,Brc1cccc(Sc2ncccc2OCCc2ccncc2)c1
3,3,3,0,Clc1ccccc1C1CC(c2cccs2)Nc2nnnn21
4,4,4,0,O=C(NCc1ccccn1)Nc1cc(C(F)(F)F)c[nH]c1=O
...,...,...,...,...
1410,15,1200,3,CCN1CCN(c2ccc(Nc3ncc(C(=O)c4cc(F)ccc4S)c(N)n3)...
1459,15,1240,3,CCNc1nc(Nc2ccc(N3CCN(C)CC3)cc2C)ncc1C(=O)c1ccc...
1400,15,1183,3,CC(O)N1CCN(c2ccc(Nc3ncc(C(=O)c4cccc(F)c4)c(N)n...
1386,15,1222,3,CNc1cc(F)c(C)c(C(=O)c2cnc(Nc3ccc(N4CCN(C)CC4)c...


In [16]:
df_augset.drop_duplicates(keep='first')

,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,C#CCOc1ccc2ccc(=O)oc2c1
1,1,1,0,CCC(N)C(O)c1cc(OC)c(Br)cc1OC
2,2,2,0,Brc1cccc(Sc2ncccc2OCCc2ccncc2)c1
3,3,3,0,Clc1ccccc1C1CC(c2cccs2)Nc2nnnn21
4,4,4,0,O=C(NCc1ccccn1)Nc1cc(C(F)(F)F)c[nH]c1=O
...,...,...,...,...
1410,15,1200,3,CCN1CCN(c2ccc(Nc3ncc(C(=O)c4cc(F)ccc4S)c(N)n3)...
1459,15,1240,3,CCNc1nc(Nc2ccc(N3CCN(C)CC3)cc2C)ncc1C(=O)c1ccc...
1400,15,1183,3,CC(O)N1CCN(c2ccc(Nc3ncc(C(=O)c4cccc(F)c4)c(N)n...
1386,15,1222,3,CNc1cc(F)c(C)c(C(=O)c2cnc(Nc3ccc(N4CCN(C)CC4)c...


In [ ]:
df_augset = df_augset.drop_duplicates(keep='first')

In [ ]:
df_augset.sort_values('src_idx')

In [ ]:
from rdkit import Chem
import random

for row in df_samp.itertuples():
    smi = row.smiles
    anc_idx = row.Index
    augs = get_augs(smi,atom_to_cnt)
    if augs:
        for aug_smi in augs:
            item = [anc_idx, anc_idx, 1, aug_smi]
            augset.append(item)

In [ ]:
df_augset = pd.DataFrame(augset, columns=['anc_idx', 'src_idx', 'aug_iter', 'smiles'])
df_augset